# Lesson 15: Evaluating LLM systems

*Companion notebook for [the lesson page](../lessons/15-evaluation.html). Worked solutions are in `solutions/15-evaluation-solutions.ipynb`; try each exercise first.*

The discipline that separates demos from products, covering golden datasets, code-based graders and LLM judges done properly, statistics that tell you whether a change really helped, an evaluation harness, regression tests, and the limits of public benchmarks.

**Runs on:** Any computer (CPU). Only the LLM-judge cells need an API key.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy anthropic pydantic python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Building a golden dataset

**How big?** Start with 20 to 50 cases on day one; a small set used daily beats a perfect set that never exists. Grow towards a few hundred. As the statistics section shows, you need hundreds before small differences become detectable.

In [ ]:
%%writefile eval_cases.jsonl
{"id": "t-001", "input": "Unit P-204 shows status ok but no data since 06:00", "expected": {"category": "data"}, "tags": ["ambiguous"]}
{"id": "t-002", "input": "Please reschedule the March calibration visit", "expected": {"category": "other"}, "tags": ["easy"]}
{"id": "q-014", "input": "How often should the desiccant be replaced at coastal sites?", "reference": "Every six months in humid climates.", "gold_chunk": "manual-x#12", "tags": ["rag", "lookup"]}
{"id": "q-031", "input": "What is the warranty period for model Z?", "reference": "NOT_IN_DOCS", "tags": ["rag", "unanswerable"]}

## Graders

### Code-based graders

Fast, free, deterministic. Use them for everything you can express as a rule.

In [ ]:
import json, re
from pydantic import BaseModel, ValidationError

def exact_match(output: str, expected: str) -> float:
    return float(output.strip().lower() == expected.strip().lower())

def contains_all(output: str, required: list[str]) -> float:
    return sum(r.lower() in output.lower() for r in required) / len(required)

def numeric_close(output: str, expected: float, rel_tol: float = 0.01) -> float:
    nums = [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", output)]
    return float(any(abs(n - expected) <= rel_tol * abs(expected) for n in nums))

def valid_schema(output: str, model: type[BaseModel]) -> float:
    try:
        model.model_validate_json(output)
        return 1.0
    except ValidationError:
        return 0.0

def cites_only_given_sources(output: str, n_sources: int) -> float:
    cited = {int(c) for c in re.findall(r"\[(\d+)\]", output)}
    return float(bool(cited) and all(1 <= c <= n_sources for c in cited))

### LLM-as-judge

- **Grade one specific criterion at a time** ("Is every claim supported by the sources?"), not "rate quality 1 to 10".
- **Prefer binary or small categorical scales** with explicit definitions over fine-grained numbers.
- **Give the judge what it needs**: the sources, the reference answer, the rubric.
- **Ask for reasoning before the verdict**, in a structured format your code can parse.
- **For pairwise comparisons, run both orders** and count only consistent wins.
- **Calibrate against humans**: have people grade 50 to 100 cases, measure agreement with the judge, and refine the rubric until agreement is high. An uncalibrated judge is a random number generator with good manners.

In [ ]:
import anthropic
client = anthropic.Anthropic()
JUDGE_MODEL = "claude-sonnet-5-5"

FAITHFULNESS_RUBRIC = """You are grading whether an answer is fully supported by source documents.

<sources>
{sources}
</sources>

<answer>
{answer}
</answer>

Go through the answer claim by claim. A claim is SUPPORTED if the sources state it or it follows
directly from them. Ignore style and completeness; judge only support.

Return JSON only:
{{"unsupported_claims": ["..."], "verdict": "PASS" or "FAIL"}}
PASS means there are no unsupported factual claims."""

def judge_faithfulness(answer: str, sources: str) -> dict:
    r = client.messages.create(
        model=JUDGE_MODEL, max_tokens=600, temperature=0,
        messages=[{"role": "user", "content": FAITHFULNESS_RUBRIC.format(sources=sources, answer=answer)},
                  {"role": "assistant", "content": "{"}],                # prefill: start the JSON
    )
    return json.loads("{" + r.content[0].text)

In [ ]:
# Needs an API key (see the setup cell)
print(judge_faithfulness(
    answer="Replace the desiccant every three months at humid sites [1]. It also protects the laser [1].",
    sources='<source id="1">Replace the cartridge every six months, or every three months at humid sites.</source>'))

## An evaluation harness

Put it together: run every case through a system variant, grade it, save everything, and summarise by slice.

In [ ]:
import json, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

def run_eval(system_fn, cases, graders, name, workers=8):
    """system_fn(input) -> output string; graders: {metric: fn(case, output) -> float in [0, 1]}"""
    def run_one(case):
        t0 = time.perf_counter()
        try:
            output, error = system_fn(case["input"]), None
        except Exception as e:
            output, error = "", repr(e)
        latency = time.perf_counter() - t0
        scores = {m: (g(case, output) if error is None else 0.0) for m, g in graders.items()}
        return {"id": case["id"], "tags": case.get("tags", []), "output": output,
                "error": error, "latency": latency, "scores": scores}

    with ThreadPoolExecutor(workers) as pool:
        results = list(pool.map(run_one, cases))

    with open(f"results_{name}.jsonl", "w") as f:              # keep every output: you will want to read them
        for r in results:
            f.write(json.dumps(r) + "\n")

    summary = defaultdict(list)
    for r in results:
        for m, s in r["scores"].items():
            summary[(m, "ALL")].append(s)
            for tag in r["tags"]:
                summary[(m, tag)].append(s)
    for (m, tag), vals in sorted(summary.items()):
        print(f"{name:<12} {m:<14} {tag:<14} {sum(vals) / len(vals):.3f}  (n={len(vals)})")
    return results

graders = {
    "category": lambda case, out: exact_match(out, case["expected"]["category"]),
}
# baseline = run_eval(classify_v3, cases, graders, "prompt_v3")
# candidate = run_eval(classify_v4, cases, graders, "prompt_v4")

In [ ]:
# A self-contained demo of the harness: 60 synthetic support messages and two rule-based "systems".
import random
random.seed(1)
TEMPLATES = {
    "hardware": ["The fan in unit {i} is grinding", "Cable to the tracker on {i} is cracked",
                 "Enclosure door on {i} will not close", "{i} power LED off after the storm"],
    "software": ["After updating {i} to 2.4 the GUI crashes", "Driver error on startup for {i}",
                 "Acquisition app freezes when exporting {i} data", "{i} firmware update failed halfway"],
    "data": ["Yesterday's values from {i} are all zero", "{i} shows ok but no data since 06:00",
             "NO2 from {i} jumps to 400 ppb at night", "{i} spectra look noisier than last week"],
    "other": ["Can we move the {i} calibration visit to March?", "Please resend the invoice for {i}",
              "Who is the site access contact for {i}?", "Request a quote for a spare unit like {i}"],
}
AMBIGUOUS = {"{i} power LED off after the storm", "{i} firmware update failed halfway",
             "{i} spectra look noisier than last week", "{i} shows ok but no data since 06:00"}
demo_cases = []
for cat, temps in TEMPLATES.items():
    for k in range(15):
        t = temps[k % len(temps)]
        demo_cases.append({"id": f"{cat}-{k}", "input": t.format(i=f"P-{random.randint(100, 299)}"),
                           "expected": {"category": cat}, "tags": ["ambiguous" if t in AMBIGUOUS else "clear"]})

def keyword_system(keywords):
    def classify(text):
        t = text.lower()
        for cat, words in keywords.items():
            if any(w in t for w in words):
                return cat
        return "other"
    return classify

v1 = keyword_system({"hardware": ["fan", "cable", "door"], "software": ["gui", "driver", "app"],
                     "data": ["values", "ppb"]})
v2 = keyword_system({"hardware": ["fan", "cable", "door", "power", "led"],
                     "software": ["gui", "driver", "app", "firmware"],
                     "data": ["values", "ppb", "no data", "spectra"]})
res_v1 = run_eval(v1, demo_cases, graders, "keywords_v1")
res_v2 = run_eval(v2, demo_cases, graders, "keywords_v2")

## Is the difference real?

**Bootstrap confidence intervals** resample your per-case scores to estimate uncertainty without formulas:

In [ ]:
import numpy as np

def bootstrap_ci(scores, n_boot=10_000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    scores = np.asarray(scores, dtype=float)
    means = rng.choice(scores, size=(n_boot, len(scores)), replace=True).mean(axis=1)
    return scores.mean(), np.quantile(means, alpha / 2), np.quantile(means, 1 - alpha / 2)

**Paired comparison**: both systems ran on the *same* cases, so compare them case by case. Most cases are answered the same way by both; what matters is the cases that changed. A paired bootstrap on per-case differences is much more sensitive than comparing two independent intervals:

In [ ]:
def paired_bootstrap(a, b, n_boot=10_000, seed=0):
    """Mean improvement of b over a, and the fraction of resamples where b is not better."""
    rng = np.random.default_rng(seed)
    d = np.asarray(b, float) - np.asarray(a, float)
    diffs = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    return d.mean(), (diffs <= 0).mean()

In [ ]:
# Is v2 really better than v1? Compare case by case.
s1 = [r["scores"]["category"] for r in res_v1]
s2 = [r["scores"]["category"] for r in res_v2]
print("v1", bootstrap_ci(s1))
print("v2", bootstrap_ci(s2))
print("paired: mean gain, fraction of resamples where v2 is not better =", paired_bootstrap(s1, s2))
fixed = [a["id"] for a, b in zip(res_v1, res_v2) if b["scores"]["category"] > a["scores"]["category"]]
broken = [a["id"] for a, b in zip(res_v1, res_v2) if b["scores"]["category"] < a["scores"]["category"]]
print("fixed:", fixed)
print("broken:", broken)

## Online evaluation

### Hands-on: an eval suite for your RAG system

Using your system from lesson 13:

1. Turn your evaluation questions into `eval_cases.jsonl` with tags (`lookup`, `comparison`, `unanswerable`, and so on). Aim for at least 40 cases.
2. Implement graders: retrieval hit (gold chunk in top 5), abstention (for `unanswerable` cases, did the answer say the documents do not contain it?), and an LLM faithfulness judge.
3. Hand-grade faithfulness for 20 answers and measure your judge's agreement with you. Refine the rubric until it agrees on at least 17 of 20.
4. Compare two variants (for example, with and without the reranker) using `run_eval` and `paired_bootstrap`. List the cases fixed and broken.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/15-evaluation.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.